# EDA и проверка препроцессинга

Здесь смотрю на данные, чтобы понять, какие проблемы есть в трейне и тесте. На основе этого собрал пайплайн в `src/preprocessing.py`
Основные моменты, которые надо было заметить:
1. Как выглядят пропуски и как они между трейном и тестом различаются
2. HTML-теги и спецсимволы в описаниях
3. Мусор в начале запросов
4. Длины текстов - описания огромные и если их не обрезать то OOM произойдет либо обучение оч долгое
5. Бренды и цвета — там тоже много дублей

In [1]:
from pathlib import Path
from collections import Counter
import re
import os
import sys
import warnings

import pandas as pd
from IPython.display import display
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)
os.environ.setdefault("TQDM_DISABLE", "1")

ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.preprocessing import clean_text, preprocess_dataframe
from src.features import normalize_color
from src.constants import RANDOM_SEED

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

DATA_DIR = ROOT / 'data'
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print(f"train shape: {train.shape}, test shape: {test.shape}")

train shape: (49496, 11), test shape: (21184, 10)


## 1. Общая картина
Распределение запросов и таргета. Важно понять, пересекаются ли юзеры (query_id) в трейне и тесте, чтобы правильно настроить валидацию

In [2]:
print("Уникальные query_id train/test:", train.query_id.nunique(), test.query_id.nunique())
print("Пересечения query_id:", len(set(train.query_id) & set(test.query_id)))

relevance_dist = train.relevance.value_counts(normalize=True).sort_index().rename("share")
print("Распределение релевантности:")
display(relevance_dist.to_frame())

print("product_locale train:")
display(train["product_locale"].value_counts(normalize=True).to_frame("share"))
print("product_locale test:")
display(test["product_locale"].value_counts(normalize=True).to_frame("share"))

Уникальные query_id train/test: 3500 1500
Пересечения query_id: 0
Распределение релевантности:


,share
relevance,
0,0.162175
1,0.064429
2,0.363039
3,0.410356


product_locale train:


,share
product_locale,
us,1.0


product_locale test:


,share
product_locale,
us,1.0


**Итого:**
* Пересечений по `query_id` нет -> можно использовать `GroupKFold`.
* Таргет смещен в сторону релевантных (классы 2 и 3).
* `product_locale` везде `us`, так что этот столбец просто мусор, можно дропать.

## 2. Плейсхолдеры и странные пропуски
В данных очень много разных вариантов пропусков: `none`, `null`, `brand_masked` и т.д. Проблема в том, что в трейне и тесте они могут быть записаны по-разному. Например, в трейне 'none', а в тесте просто пустая строка. Надо это унифицировать

In [3]:
markers = ["none", "None", "[null]", "null", "unknown_brand", "brand_masked", "generic", "", "0", "1"]
cols = ["product_description", "product_bullet_point", "product_brand", "product_color"]

def marker_stats(col: str) -> pd.DataFrame:
    total_train, total_test = len(train), len(test)
    c_train = Counter(train[col].fillna(""))
    c_test = Counter(test[col].fillna(""))
    rows = []
    for marker in markers:
        cnt_train = c_train.get(marker, 0)
        cnt_test = c_test.get(marker, 0)
        if cnt_train == 0 and cnt_test == 0:
            continue
        rows.append(
            {
                "marker": marker or "<empty>",
                "train_count": cnt_train,
                "train_pct": round(cnt_train / total_train * 100, 2),
                "test_count": cnt_test,
                "test_pct": round(cnt_test / total_test * 100, 2),
            }
        )
    return pd.DataFrame(rows).sort_values("train_pct", ascending=False).reset_index(drop=True)

placeholder_tables = {col: marker_stats(col) for col in cols}
display(pd.concat(placeholder_tables, names=["column", "row"]).reset_index(level=0))

,column,marker,train_count,train_pct,test_count,test_pct
row,,,,,,
0,product_description,none,24979,50.47,0,0.00
1,product_description,1,37,0.07,0,0.00
2,product_description,0,7,0.01,0,0.00
3,product_description,<empty>,1,0.00,10890,51.41
0,product_bullet_point,<empty>,19222,38.84,2996,14.14
0,product_brand,unknown_brand,9891,19.98,0,0.00
1,product_brand,<empty>,2091,4.22,1266,5.98
2,product_brand,brand_masked,0,0.00,2065,9.75
0,product_color,<empty>,15293,30.90,6740,31.82


## 3. HTML и спецсимволы
Описание товаров часто парсят со страниц в интернете как есть, поэтому там есть теги и `&amp;`. Если это не почистить, то результатом токенайзера будет много шумных токенов

In [4]:
html_re = re.compile(r"<[^>]+>")
entity_re = re.compile(r"&[a-zA-Z]+;|&#\d+;")


def html_summary(col: str) -> dict:
    col_train = train[col].fillna("")
    col_test = test[col].fillna("")
    return {
        "column": col,
        "train_html_%": round(col_train.str.contains(html_re).mean() * 100, 2),
        "test_html_%": round(col_test.str.contains(html_re).mean() * 100, 2),
        "train_entity_%": round(col_train.str.contains(entity_re).mean() * 100, 2),
        "test_entity_%": round(col_test.str.contains(entity_re).mean() * 100, 2),
    }

html_stats = pd.DataFrame([html_summary(col) for col in ["product_title", "product_description", "product_bullet_point"]])
print("Доля строк с HTML/сущностями (%):")
display(html_stats)

# Пример того, как работает моя функция clean_text
sample_html = train.loc[train["product_description"].fillna("").str.contains(html_re)].head(2)
sample_html = sample_html[["product_description"]].assign(cleaned=lambda df: df["product_description"].apply(clean_text))
display(sample_html)

Доля строк с HTML/сущностями (%):


,column,train_html_%,test_html_%,train_entity_%,test_entity_%
0,product_title,0.00,0.00,0.01,0.03
1,product_description,28.30,26.44,0.24,0.19
2,product_bullet_point,0.22,0.23,0.33,0.41


,product_description,cleaned
0,<p>bila print flowy sleeveless rayon dress per...,bila print flowy sleeveless rayon dress perfec...
2,<b>unisex funny men 3d lifelike animal paw soc...,unisex funny men 3d lifelike animal paw socks ...


В описании ~30% HTML

## 4. Запросы (Query)
В начале строки иногда встречаются лишние символы (точки, цифры). Это может сбить BM25 или эмбеддинги

In [5]:
starts_punct = train["query"].str.match(r"^\W").mean()
starts_digit = train["query"].str.match(r"^\d").mean()
lengths = train["query"].str.len()

summary = pd.DataFrame(
    {
        "metric": ["starts_with_punct_%", "starts_with_digit_%", "mean_len", "median_len", "max_len"],
        "value": [
            round(starts_punct * 100, 2),
            round(starts_digit * 100, 2),
            round(lengths.mean(), 1),
            lengths.median(),
            lengths.max(),
        ],
    }
)
display(summary)

,metric,value
0,starts_with_punct_%,0.52
1,starts_with_digit_%,5.75
2,mean_len,19.80
3,median_len,19.00
4,max_len,138.00


## 5. Длинные тексты и обрезка
Описания бывают очень длинными (до 3000+ символов), а релевантная инфа часто размазана.
Я использую стратегию `_extract_relevant_span` (в `preprocessing.py`): ищу кусок текста, который лучше всего матчится с запросом, и оставляю только его + окрестность. Остальное режем до 256 символов

In [6]:
raw_len = pd.DataFrame(
    {
        "column": ["product_description", "product_bullet_point"],
        "gt_256_%": [
            round((train["product_description"].fillna("").str.len() > 256).mean() * 100, 2),
            round((train["product_bullet_point"].fillna("").str.len() > 256).mean() * 100, 2),
        ],
        "empty_%": [
            round((train["product_description"].fillna("").str.len() == 0).mean() * 100, 3),
            round((train["product_bullet_point"].fillna("").str.len() == 0).mean() * 100, 3),
        ],
        "mean_len": [
            round(train["product_description"].fillna("").str.len().mean(), 1),
            round(train["product_bullet_point"].fillna("").str.len().mean(), 1),
        ],
        "median_len": [
            train["product_description"].fillna("").str.len().median(),
            train["product_bullet_point"].fillna("").str.len().median(),
        ],
    }
)

sample = train.sample(n=20000, random_state=RANDOM_SEED)
processed = preprocess_dataframe(sample)
rows = []
for col in ["product_description", "product_bullet_point"]:
    orig = sample[col].fillna("").str.len()
    new = processed[col].fillna("").str.len()
    rows.append(
        {
            "column": col,
            "share_reduced_%": round((new < orig).mean() * 100, 2),
            "share_empty_after_%": round((new == 0).mean() * 100, 2),
            "mean_before": round(orig.mean(), 1),
            "mean_after": round(new.mean(), 1),
        }
    )
postproc_stats = pd.DataFrame(rows)

# Пример "живого" кейса
long_row = train.loc[train["product_description"].fillna("").str.len() > 800].iloc[0]
processed_row = preprocess_dataframe(pd.DataFrame([long_row])).iloc[0]
span_example = pd.DataFrame(
    {
        "query": [long_row["query"]],
        "raw_desc_len": [len(str(long_row["product_description"]))],
        "processed_len": [len(processed_row["product_description"])],
        "processed_head": [processed_row["product_description"][:220]],
    }
)

print("До предобработки (train):")
display(raw_len)
print("\nПосле `_extract_relevant_span` + обрезки (сэмпл 20k):")
display(postproc_stats)
print("\nПример: длинное описание -> релевантное окно")
display(span_example)

До предобработки (train):


,column,gt_256_%,empty_%,mean_len,median_len
0,product_description,37.64,0.002,345.5,4.0
1,product_bullet_point,47.94,38.835,422.6,222.0



После `_extract_relevant_span` + обрезки (сэмпл 20k):


,column,share_reduced_%,share_empty_after_%,mean_before,mean_after
0,product_description,90.02,50.44,346.2,100.3
1,product_bullet_point,48.44,38.70,423.8,129.0



Пример: длинное описание -> релевантное окно


,query,raw_desc_len,processed_len,processed_head
0,"#do disturb, jeidah bila",870,256,unisex funny men 3d lifelike animal paw socks ...


Получилось сократить среднюю длину описания с ~350 до ~100. Это сильно ускорит инференс и обучение.

## 6. Нормализация брендов и цветов
Бренды и цвета очень "грязные". Много вариантов написания одного и того же, много мусора типа `unknown`.
В `src/features.py` есть нормализация цветов (сведение к базовым black/white/red...).

In [7]:
brands = train["product_brand"].fillna("").str.lower()
brand_markers = {"unknown", "unknown_brand", "generic", "brand_masked", "none", ""}
brand_unknown_frac = brands.isin(brand_markers).mean() * 100

colors_raw = train["product_color"].fillna("")
colors_norm = colors_raw.apply(normalize_color)
color_unknown_frac = colors_norm.isin({"unknown", ""}).mean() * 100

brand_stats = pd.DataFrame(
    {
        "metric": ["unknown_share_%", "unique_raw_brands"],
        "value": [round(brand_unknown_frac, 2), brands.nunique()],
    }
)
color_stats = pd.DataFrame(
    {
        "metric": ["unknown_share_%", "unique_raw_colors", "unique_normalized_colors"],
        "value": [round(color_unknown_frac, 2), colors_raw.nunique(), colors_norm.nunique()],
    }
)

print("Бренды:")
display(brand_stats)
print("\nЦвета (нормализованные топ-10):")
display(colors_norm.value_counts().head(10))
print("\nЦвета (статистика):")
display(color_stats)

Бренды:


,metric,value
0,unknown_share_%,24.37
1,unique_raw_brands,18384.00



Цвета (нормализованные топ-10):


product_color
           15293
black       8457
white       3543
blue        2308
gray        1962
unknown     1645
red         1428
silver      1413
brown        972
pink         909
Name: count, dtype: int64


Цвета (статистика):


,metric,value
0,unknown_share_%,34.22
1,unique_raw_colors,10962.00
2,unique_normalized_colors,3154.00


## 7. Длины текстов после очистки
Чтобы выбрать max_length для моделей, полезно видеть квантили длин после `preprocess_dataframe`

In [8]:
text_cols = ['query', 'product_title', 'product_description', 'product_bullet_point']

def length_quantiles(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        lengths = df[col].fillna('').str.len()
        rows.append(
            {
                'column': col,
                'p50': lengths.quantile(0.5),
                'p90': lengths.quantile(0.9),
                'p95': lengths.quantile(0.95),
                'p99': lengths.quantile(0.99),
                'max': lengths.max(),
            }
        )
    return pd.DataFrame(rows).round(1)


raw_quantiles = length_quantiles(sample, text_cols)
print('Длины до clean/усечения (сэмпл 20k):')
display(raw_quantiles)

print('Длины после preprocess_dataframe (сэмпл 20k):')
display(length_quantiles(processed, text_cols))

Длины до clean/усечения (сэмпл 20k):


,column,p50,p90,p95,p99,max
0,query,19.0,30.0,34.0,46.0,138
1,product_title,121.0,229.0,248.0,283.0,507
2,product_description,4.0,1180.0,1416.0,1599.0,2898
3,product_bullet_point,225.0,1170.0,1417.0,1961.0,2485


Длины после preprocess_dataframe (сэмпл 20k):


,column,p50,p90,p95,p99,max
0,query,19.0,30.0,34.0,46.0,138
1,product_title,121.0,229.0,248.0,283.0,507
2,product_description,0.0,256.0,256.0,256.0,256
3,product_bullet_point,169.0,256.0,256.0,256.0,256


До очистки хвосты огромные: p99 desc ~1600 символов, bullet ~1960 (max до 2.5–2.9k). После `preprocess_dataframe` оба поля обрезаны до 256, медиана desc → 0. max_length=256 спасает от хвостов, оптимальный по скору, быстрый для инференса - поэтому оставил